# HotpotQA Knowledge Network Demo

This notebook is the **full HotpotQA-specific knowledge-network demo path** for the GKN repo.

It compares:
- embedding baseline retrieval
- embedding + HotpotQA-specific KN retrieval

The HotpotQA graph design uses:
- title entities (article anchors)
- named entities (multi-hop bridge nodes)
- corpus-only entity-sharing edges between chunks
- graph path explanations for interpretability
- reusable and reproducible vector-store caching

> **Fair benchmark:** the retrieval graph is built from observable corpus text
> only. Gold supporting facts are used **exclusively** to score relevance, never
> to build the graph the retriever uses, so the baseline-vs-hybrid comparison is
> honest (no answer-key leakage).
>
> **Data:** the HotpotQA dataset is downloaded and cached automatically on first
> run (see Section 2); subsequent runs load the local copy.

## 1. Methodology overview

This benchmark follows the same overall philosophy as the synthetic-document KN demo, but adapts the graph structure to the HotpotQA use case.

### What makes this KN benchmark-specific?

HotpotQA is a multi-hop benchmark where questions often require bridging across multiple articles. Therefore, the graph for this use case focuses on:

- article titles as graph anchors
- named entities as bridge nodes for multi-hop connections
- corpus-only entity-sharing edges between chunks
- graph-expanded chunk retrieval
- path explanations for interpretability

### What is being compared?

1. **Embedding baseline retrieval**
2. **Embedding retrieval + HotpotQA-specific knowledge network**

### What counts as relevance here?

Relevance is approximated by mapping the gold supporting titles to normalized
document IDs and treating chunks from those documents as relevant. This is a
coarse, title-level approximation, but it gives a consistent target for both
retrievers.

### Avoiding answer-key leakage (important)

The gold supporting facts are used **only** to define this relevance target for
scoring. They are deliberately **not** encoded into the retrieval graph. An
earlier version added `SupportingFact` nodes and `SUPPORTS` edges (built from the
gold labels) and then rewarded the hybrid retriever for landing near them — that
leaks the answer key into the retriever and makes any hybrid gain untrustworthy.
The graph is now built from observable corpus text only, so the comparison is
fair. Expect honest results: the hybrid retriever should be judged on whether it
genuinely improves retrieval, not on label proximity.

## 2. Setup and configuration

This step loads your local configuration from `.env`, prepares artifact folders, and prints the active benchmark settings.

### Runs on both Mac and PC

Keep `.env` **path-free** — do not set an absolute `HOTPOTQA_DATA_PATH`. By default the dataset path resolves *relative to the repo* (`data/hotpot_train_v1.1.json`), so the same checkout works on macOS and Windows. (An absolute Windows path in `.env` will not resolve on a Mac.) See `.env.example`.

### Embeddings: Azure-preferred with automatic local fallback

- `EMBEDDING_CHOICE=small` (or `large`) uses your Azure / OpenAI-compatible endpoint.
- If the cloud path is unavailable — `openai` not installed, no `OPENAI_API_KEY`, or a failed call — the store **automatically falls back to a local `sentence-transformers` model** (controlled by `EMBEDDING_FALLBACK_TO_LOCAL`, default `true`).
- The cell below prints the *configured* choice; Section 4 prints the mode **actually in effect** after any fallback.
- `faiss` is **optional** — without it, an exact NumPy index is used.

### Dataset download and caching

The HotpotQA dataset is fetched automatically the first time you load it (Section 3) and cached at `config.hotpotqa_data_path`. Later runs load the local copy directly. The loader tries the official CMU mirror first and falls back to the Hugging Face mirror automatically. The file is large (~570 MB for the full train split) and is git-ignored.

### Reproducibility notes

- the HotpotQA subset is selected with a fixed random seed
- the vector-store cache key includes sample size, random seed, **effective** embedding mode, model, and chunk signature — so an Azure run and a local-fallback run are cached separately and never collide

### Recommended first-run settings

- `HOTPOTQA_SAMPLE_SIZE=25` (raise to 100+ once it runs cleanly; large samples are slow on CPU local embeddings)
- `FORCE_REBUILD_VECTOR_STORE=false` unless you intentionally want a rebuild


In [ ]:
from pathlib import Path
import sys
import pandas as pd

sys.path.append(str(Path.cwd().parent / 'src'))

from geometric_knowledge_network.config import GKNConfig
from geometric_knowledge_network.hotpotqa_benchmark import HotpotBenchmarkRunner
from geometric_knowledge_network.hotpotqa_graph import HotpotEntityExtractor, HotpotGraphBuilder
from geometric_knowledge_network.hotpotqa_loader import HotpotQALoader
from geometric_knowledge_network.hybrid_retriever import HybridRetriever
from geometric_knowledge_network.multihop_retriever import MultiHopRetriever
from geometric_knowledge_network.multihop_benchmark import MultiHopBenchmarkRunner
from geometric_knowledge_network.path_explainer import GraphPathExplainer
from geometric_knowledge_network.reporting import ArtifactManager
from geometric_knowledge_network.vector_store import EmbeddingVectorStore
from geometric_knowledge_network.visualization import draw_subgraph

config = GKNConfig()
artifacts = ArtifactManager(config.artifacts_dir)
run_id = artifacts.timestamp()

def status(message: str):
    print(f'[DONE] {message}')

status(f'Configuration loaded. Run ID: {run_id}')
print('HotpotQA path:', config.hotpotqa_data_path)
print('Path exists:', Path(config.hotpotqa_data_path).exists())
print('Sample size:', config.hotpotqa_sample_size)
print('Random seed:', config.hotpotqa_random_seed)
print('Chunk size / overlap:', config.hotpotqa_chunk_size, '/', config.hotpotqa_chunk_overlap)
print('Embedding choice:', config.embedding_choice)
print('Embedding model:', config.openai_embedding_model if config.embedding_choice != 'local' else config.local_embedding_model)
print('Force rebuild vector store:', config.force_rebuild_vector_store)
print('Checkpoints dir:', config.checkpoints_dir)
print('Base URL set:', bool(config.openai_base_url))
print('API key set:', bool(config.openai_api_key))

## 3. Load HotpotQA sample and construct retrieval units

This step converts sampled HotpotQA contexts into document and chunk units suitable for both vector retrieval and graph construction.

### Why this matters

The same chunk corpus is used by:
- the embedding baseline
- the HotpotQA-specific knowledge network
- the retrieval benchmark evaluation

This is important for fairness: both retrievers operate over the same underlying retrieval units.

In [ ]:
loader = HotpotQALoader()
documents, chunks, samples = loader.load_samples(
    config.hotpotqa_data_path,
    sample_size=config.hotpotqa_sample_size,
    random_seed=config.hotpotqa_random_seed,
    chunk_size=config.hotpotqa_chunk_size,
    chunk_overlap=config.hotpotqa_chunk_overlap,
)
status(f'Loaded {len(samples)} HotpotQA samples, {len(documents)} documents, and {len(chunks)} chunks.')

for sample in samples[:3]:
    print('-' * 100)
    print('Question ID:', sample.question_id)
    print('Question:', sample.question)
    print('Answer:', sample.answer)
    print('Supporting titles:', sample.supporting_titles)

## 4. Build or load the embedding vector store

This step either:
- loads a previously built vector store from cache, or
- builds a new one and saves it for reuse

### Why this matters

The HotpotQA benchmark should be reusable and reproducible. If you rerun the notebook with the same:
- sample size
- random seed
- embedding mode
- embedding model
- chunk corpus

then the vector store should be reused automatically rather than rebuilt.

If you hit rate limits in cloud embedding mode, this cached reuse becomes especially important.

In [ ]:
vector_store = EmbeddingVectorStore(config)
vector_store.build(chunks)
# `mode` is the embedding mode actually in effect. It equals EMBEDDING_CHOICE when
# the cloud provider is available, and automatically falls back to 'local' if the
# openai package / API key is missing or a cloud call fails.
print('Effective embedding mode:', vector_store.mode, '| model:', vector_store._embedding_model_name())
status('Embedding vector store ready (built or loaded from cache).')

## 5. Build the HotpotQA-specific knowledge network

This step builds the graph structure used by the KN-enhanced retriever.

### Graph design for HotpotQA

The graph includes:
- document nodes
- chunk nodes
- title entity nodes
- named entity nodes
- concept nodes
- `SHARES_ENTITY` edges linking chunks that mention the same distinctive named entity (the multi-hop bridges)

This differs from the governance demo because the benchmark structure is different: HotpotQA is about multi-hop article/entity bridging, not requirement-control logic.

> **No gold labels in the graph.** Unlike an earlier version, the builder does
> **not** receive the HotpotQA samples and does **not** create supporting-fact
> nodes. The graph is purely corpus-derived, which keeps the benchmark fair (see
> Section 1).

In [ ]:
extractor = HotpotEntityExtractor()
# IMPORTANT (benchmark fairness): the graph is built from the corpus only
# (documents, chunks, and entities extracted from chunk text). Gold supporting
# facts are deliberately NOT passed in here -- they are used only to define
# evaluation relevance (see HotpotRelevanceMapper). Baking them into the graph
# would leak the answer key into the retriever.
graph = HotpotGraphBuilder(extractor).build(documents, chunks)
status(f'HotpotQA knowledge network built with {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.')

node_type_counts = pd.Series([graph.nodes[node].get('node_type', 'Unknown') for node in graph.nodes]).value_counts().reset_index()
node_type_counts.columns = ['node_type', 'count']
display(node_type_counts)

## 6. Evaluate baseline vs KN-enhanced retrieval

This step runs the benchmark comparison.

### What is being measured?

For each HotpotQA question, we compare:
- baseline embedding retrieval
- embedding retrieval + HotpotQA-specific KN augmentation

and compute retrieval metrics such as:
- hit rate
- recall@k
- precision@k
- MRR

We also capture path explanations when graph expansion adds results.

In [ ]:
hybrid = HybridRetriever(vector_store, graph)
path_explainer = GraphPathExplainer(graph)
benchmark = HotpotBenchmarkRunner(vector_store, hybrid, chunks, path_explainer)
benchmark_result = benchmark.run(samples, top_k=config.top_k, graph_hops=config.graph_hops)
status(f'Completed HotpotQA KN benchmark for {len(samples)} questions.')

evaluation_df = benchmark_result.evaluation_df
aggregate_df = benchmark_result.aggregate_df
paths_df = benchmark_result.paths_df

display(evaluation_df.head())
display(aggregate_df)
display(paths_df.head())

## 6b. Headline comparison — baseline vs hybrid vs multi-hop (PPR)

This is the demonstration that matters for **agentic RAG**: three retrievers over the *same* corpus and graph, scored at a larger budget (`top_k=10`) with **document-level multi-hop metrics**:

- `doc_recall` — fraction of supporting documents covered
- `all_docs_hit` — `1.0` only when **every** supporting document is recovered, including the hard second "bridge" hop that is often semantically distant from the query

The **multi-hop** retriever keeps the dense top hits in place — so MRR is preserved (*add-not-demote*) — and uses **Personalized PageRank**, seeded on the hop-1 chunks and the query-matching entities adjacent to them, to pull in bridge chunks from documents the vector search missed. This is where the knowledge network earns its keep.

> Read it as: `doc_recall` / `all_docs_hit` should **rise** for `multihop` while `mrr` stays close to `baseline`. The old `hybrid` typically ties on coverage but **drops MRR** (it reshuffles strong hits) — the problem the PPR scorer fixes.

In [ ]:
# Headline comparison: baseline vs hybrid (old) vs multi-hop (PPR, add-not-demote).
multihop = MultiHopRetriever(vector_store, graph)
mh_runner = MultiHopBenchmarkRunner(vector_store, hybrid, multihop, chunks)

COMPARISON_TOP_K = 10  # larger budget leaves room for graph-discovered bridge docs
mh_result = mh_runner.run(samples, top_k=COMPARISON_TOP_K, graph_hops=config.graph_hops)
status(f'Completed 3-way retrieval comparison (top_k={COMPARISON_TOP_K}).')

multihop_eval_df = mh_result.evaluation_df
multihop_aggregate_df = mh_result.aggregate_df

print('Aggregate metrics by retriever (higher is better). '
      'Watch doc_recall / all_docs_hit (multi-hop recovery) vs mrr (ranking quality):')
display(multihop_aggregate_df)

mh_aggregate_path = artifacts.save_dataframe(multihop_aggregate_df, f'hotpotqa_multihop_aggregate_{run_id}.csv')
mh_query_path = artifacts.save_dataframe(multihop_eval_df, f'hotpotqa_multihop_query_level_{run_id}.csv')
status(f'Saved multi-hop comparison to {mh_aggregate_path}')

## 7. Inspect retrieval examples and graph-path explanations

This step helps interpret how the KN behaves on individual examples.

For the first few examples, we want to inspect:
- the question
- the supporting titles
- baseline retrieval metrics
- hybrid retrieval metrics
- whether graph-expanded paths were used


In [ ]:
for sample in samples[:3]:
    print('=' * 100)
    print('Question ID:', sample.question_id)
    print('Question:', sample.question)
    print('Supporting titles:', sample.supporting_titles)
    sample_rows = evaluation_df[evaluation_df['question_id'] == sample.question_id]
    display(sample_rows)
    sample_paths = paths_df[paths_df['question_id'] == sample.question_id]
    if not sample_paths.empty:
        print('Graph-expanded paths:')
        display(sample_paths)
    else:
        print('No graph-expanded paths recorded for this example.')

## 8. Visualize the knowledge network locally

This step renders a local subgraph around one chunk so you can inspect how the benchmark-specific KN is structured.

In [ ]:
center_chunk_id = chunks[0].chunk_id if chunks else None
if center_chunk_id:
    figure_path = artifacts.figures_dir / f'hotpotqa_kn_subgraph_{run_id}.png'
    draw_subgraph(graph, center_chunk_id, radius=2, save_path=figure_path)
    status(f'HotpotQA KN subgraph rendered and saved to {figure_path}')
else:
    print('No chunks available for graph visualization.')

## 9. Save benchmark artifacts

This step saves:
- query-level retrieval metrics
- aggregate retrieval metrics
- graph-path explanation outputs
- graph summary


In [ ]:
query_level_path = artifacts.save_dataframe(evaluation_df, f'hotpotqa_kn_query_level_{run_id}.csv')
aggregate_path = artifacts.save_dataframe(aggregate_df, f'hotpotqa_kn_aggregate_{run_id}.csv')
paths_path = artifacts.save_dataframe(paths_df, f'hotpotqa_kn_paths_{run_id}.csv')
graph_summary_path = artifacts.save_graph_summary(graph, f'hotpotqa_kn_graph_summary_{run_id}.json')

status(f'Saved HotpotQA KN query-level results to {query_level_path}')
status(f'Saved HotpotQA KN aggregate results to {aggregate_path}')
status(f'Saved path explanations to {paths_path}')
status(f'Saved graph summary to {graph_summary_path}')

## 10. Interpretation notes

When interpreting the results, keep in mind:

- all retrievers use the **same** chunk corpus and graph; the graph is built from corpus text only (gold facts are used solely to define relevance), so the comparison is leakage-free
- the chunk-level metrics in Section 6 (`hit_rate`, `recall@k`, `precision@k`, `MRR`) are dominated by hop-1; strong embeddings already do well there
- the **document-level metrics in Section 6b** (`doc_recall`, `all_docs_hit`) are the real test of the knowledge network: do we recover the second **bridge** document, the one the query does not point at directly?
- the **multi-hop (PPR)** retriever is designed to *add* bridge documents without demoting the dense top hit, so look for `all_docs_hit` / `doc_recall` going **up** while `MRR` stays close to `baseline`
- the old **hybrid** scorer is kept only as a contrast — it tends to drop MRR because it reshuffles strong hits with a query-agnostic bonus

### What to try next (research direction)

- **sentence-level** supporting-fact relevance instead of title-level (finer headroom)
- a true **agentic loop**: retrieve → extract entities → expand along the graph → re-retrieve, so the graph proposes the next hop
- sweep **KN structure** knobs (`HOTPOTQA_CHUNK_SIZE`, `HOTPOTQA_CHUNK_OVERLAP`, and entity-granularity settings in `GraphRetrievalConfig`) **against these multi-hop metrics**
- richer **geometry**: knowledge-graph embeddings (RotatE/ComplEx), hyperbolic embeddings for hierarchy, GNN node embeddings, late-interaction retrieval — see the README *Theoretical formulation* section

## 11. Final note

This notebook is the **HotpotQA-specific KN demo path** for the repo.

Suggested progression:
- `HOTPOTQA_SAMPLE_SIZE=25` for first validation
- `HOTPOTQA_SAMPLE_SIZE=50` for a stronger smoke benchmark
- `HOTPOTQA_SAMPLE_SIZE=100` or more for a more meaningful public comparison run

Suggested embedding progression:
- `EMBEDDING_CHOICE=local` for fast debugging and zero API quota risk
- `EMBEDDING_CHOICE=small` for cloud benchmark runs
- `EMBEDDING_CHOICE=large` only when higher-quality embeddings are worth the extra cost/quota

If you reuse the same sample size, seed, chunk corpus, and embedding setup, the vector store should be loaded from cache automatically.